# Retrieval Quality Probe

Diagnostic for the SSOC RAG embedding/retrieval layer. Tests whether the right chunk is surfaced for a hand-curated set of queries, **before** we wire up generation in Phase 3.

This isolates retrieval failure from generation failure. If retrieval is poor, no prompt engineering can save the system; we'd need to fix chunking, embeddings, or retrieval parameters first.

**Retrieval is deterministic** — embedding model has no temperature, HNSW returns identical top-k for the same query vector once built, chunks are static. Re-running this notebook without changes yields identical results (verified in the determinism cell at the end).

In [1]:
"""Setup: load env, locate the repo root so `import src.*` works from notebooks/."""
import json
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from src.embed import embed_query  # noqa: E402
from src.index import load_index   # noqa: E402

col = load_index()
print(f"collection count: {col.count()}  (expect 1475)")

collection count: 1475  (expect 1475)


## Probe set (25 queries)

- **Answerable (15)**: each has a known gold SSOC code; tests if cosine retrieval surfaces it. Mix of phrasing (keyword, definition, paraphrase) and target depth (4-digit unit group + 5-digit detailed occupation). Includes a synonym test ("blockchain developer" → 25121 via the `ex_under` examples).
- **Unanswerable (6)**: SSIC confusion, salary attribute (not in SSOC), fabricated code, obsolete version, off-topic. Expected behaviour: high top-1 distance vs. answerable.
- **Edge (4)**: one-word ambiguous, mild typo, niche, broad category. Documented for behavioural understanding; not pass/fail.

In [2]:
# (query, gold, category)
PROBES = [
    # --- answerable (15) ---
    ("Which SSOC code is for a software developer?",                              "25121", "answerable"),
    ("What is the SSOC code for an electrician?",                                 "74110", "answerable"),
    ("Which SSOC code is for a taxi driver?",                                     "83221", "answerable"),
    ("What's the code for a registered nurse?",                                   "22200", "answerable"),
    ("Which SSOC code covers a cook in a restaurant?",                            "51201", "answerable"),
    ("What is the SSOC code for an accountant?",                                  "24111", "answerable"),
    ("What SSOC code applies to a blockchain developer?",                         "25121", "answerable"),
    ("Which SSOC code applies to a Member of Parliament?",                        "11110", "answerable"),
    ("What's the SSOC code for a hair stylist?",                                  "51411", "answerable"),
    ("SSOC code for a chef in a restaurant",                                      "34341", "answerable"),
    ("Which code covers a plumber?",                                              "71261", "answerable"),
    ("What SSOC code is for a building painter?",                                 "71311", "answerable"),
    ("Code for a secretary handling correspondence and scheduling",               "41201", "answerable"),
    ("Person who develops computer software applications",                        "25121", "answerable"),
    ("What is the SSOC unit group for software and applications developers?",     "2512",  "answerable"),
    # --- unanswerable (6) ---
    ("What SSIC code applies to the food and beverage retail industry?",          None,    "unanswerable"),
    ("What is the average monthly salary of a software developer?",               None,    "unanswerable"),
    ("What occupation does SSOC code 88888 refer to?",                            None,    "unanswerable"),
    ("What was the SSOC code for IT support in SSOC 2010?",                       None,    "unanswerable"),
    ("Tell me a joke about taxes.",                                               None,    "unanswerable"),
    ("What is the population of Singapore?",                                      None,    "unanswerable"),
    # --- edge / informative (4) ---
    ("doctor",                                                                    None,    "edge"),
    ("electrcian",                                                                None,    "edge"),
    ("Person who designs user interfaces for mobile apps",                        None,    "edge"),
    ("machine operator",                                                          None,    "edge"),
]
n_ans = sum(1 for _, g, _ in PROBES if g)
n_un  = sum(1 for _, g, c in PROBES if g is None and c == 'unanswerable')
n_eg  = sum(1 for _, _, c in PROBES if c == 'edge')
print(f"{len(PROBES)} probes loaded ({n_ans} answerable, {n_un} unanswerable, {n_eg} edge)")

25 probes loaded (15 answerable, 6 unanswerable, 4 edge)


## Run retrieval

For each probe: embed the query once, query Chroma for top-5 with distances and metadata, collect into a DataFrame.

In [3]:
import pandas as pd

K = 5
rows = []
for q, gold, category in PROBES:
    res = col.query(
        query_embeddings=[embed_query(q)],
        n_results=K,
        include=["documents", "metadatas", "distances"],
    )
    ids = res["ids"][0]
    distances = res["distances"][0]
    documents = res["documents"][0]
    metadatas = res["metadatas"][0]
    titles = [m.get("title", "") for m in metadatas]
    rank = ids.index(gold) + 1 if gold and gold in ids else None
    rows.append({
        "query": q,
        "category": category,
        "gold": gold,
        "gold_rank": rank,
        "hit@1": bool(gold and ids[0] == gold),
        "hit@4": bool(gold and gold in ids[:4]),
        "hit@5": bool(gold and gold in ids),
        "top1_distance": round(distances[0], 4),
        "top5_ids": ids,
        "top5_titles": titles,
        "top5_distances": [round(d, 4) for d in distances],
        "_docs": documents,
    })
df = pd.DataFrame(rows)
print(f"retrieved top-{K} for {len(df)} probes")
df.drop(columns=["_docs", "top5_titles", "top5_distances"])

retrieved top-5 for 25 probes


,query,category,gold,gold_rank,hit@1,hit@4,hit@5,top1_distance,top5_ids
0,Which SSOC code is for a software developer?,answerable,25121,1.0,True,True,True,0.2189,"[25121, 2519, 2512, 25141, 25190]"
1,What is the SSOC code for an electrician?,answerable,74110,1.0,True,True,True,0.2011,"[74110, 7411, 7412, 74121, 74133]"
2,Which SSOC code is for a taxi driver?,answerable,83221,1.0,True,True,True,0.2040,"[83221, 8322, 83229, 9331, 83312]"
3,What's the code for a registered nurse?,answerable,22200,1.0,True,True,True,0.2700,"[22200, 2220, 32200, 3220, 53201]"
4,Which SSOC code covers a cook in a restaurant?,answerable,51201,1.0,True,True,True,0.2168,"[51201, 5120, 94103, 3434, 51202]"
5,What is the SSOC code for an accountant?,answerable,24111,1.0,True,True,True,0.2228,"[24111, 2411, 24113, 3313, 43119]"
6,What SSOC code applies to a blockchain developer?,answerable,25121,1.0,True,True,True,0.2273,"[25121, 25112, 25245, 25190, 2519]"
7,Which SSOC code applies to a Member of Parliam...,answerable,11110,3.0,False,True,True,0.2623,"[1111, 1114, 11110, X5000, 11140]"
8,What's the SSOC code for a hair stylist?,answerable,51411,1.0,True,True,True,0.2038,"[51411, 5141, 51419, 51412, 51421]"
9,SSOC code for a chef in a restaurant,answerable,34341,2.0,False,True,True,0.2142,"[3434, 34341, 51201, 5120, 94103]"


## Headline metrics (answerable subset)

In [4]:
ans = df[df["category"] == "answerable"]
n = len(ans)
r1 = ans["hit@1"].sum() / n
r4 = ans["hit@4"].sum() / n
r5 = ans["hit@5"].sum() / n
mrr = ans["gold_rank"].apply(lambda r: (1.0 / r) if r else 0.0).mean()

print(f"answerable probes:  {n}")
print(f"recall@1:           {int(r1*n)}/{n} = {100*r1:.0f}%")
print(f"recall@4:           {int(r4*n)}/{n} = {100*r4:.0f}%")
print(f"recall@5:           {int(r5*n)}/{n} = {100*r5:.0f}%")
print(f"MRR:                {mrr:.3f}")

answerable probes:  15
recall@1:           11/15 = 73%
recall@4:           15/15 = 100%
recall@5:           15/15 = 100%
MRR:                0.844


## Top-1 distance distribution by category

If answerable distances are clearly < unanswerable distances, the absolute distance is a candidate threshold for abstention later. If they overlap heavily, only prompt-based abstention will work.

In [5]:
for cat in ["answerable", "unanswerable", "edge"]:
    sub = df[df["category"] == cat]["top1_distance"]
    print(f"{cat:13s} n={len(sub):2d}  "
          f"min={sub.min():.4f}  median={sub.median():.4f}  "
          f"mean={sub.mean():.4f}  max={sub.max():.4f}")

answerable    n=15  min=0.1740  median=0.2177  mean=0.2251  max=0.2828
unanswerable  n= 6  min=0.2697  median=0.3560  mean=0.3490  max=0.4264
edge          n= 4  min=0.3316  median=0.3371  mean=0.3417  max=0.3611


## Per-probe chunk dump

Every probe, every top-5 result, with the **full chunk text** verbatim. This is what the retrieval layer actually returns — the ground truth for "what could the LLM have seen".

In [6]:
for i, row in df.iterrows():
    print("=" * 100)
    print(f"PROBE {i+1:2d}  category={row['category']}  gold={row['gold']}  "
          f"gold_rank={row['gold_rank']}  hit@1={row['hit@1']}  hit@4={row['hit@4']}")
    print(f"Q: {row['query']}")
    print("-" * 100)
    for rank, (id_, dist, title, doc) in enumerate(
        zip(row["top5_ids"], row["top5_distances"], row["top5_titles"], row["_docs"]),
        start=1,
    ):
        print(f"\n  [rank {rank}  distance {dist:.4f}  id={id_}  title={title}]")
        body = doc.replace("\n", "\n  ")
        print("  " + body)
    print()

PROBE  1  category=answerable  gold=25121  gold_rank=1.0  hit@1=True  hit@4=True
Q: Which SSOC code is for a software developer?
----------------------------------------------------------------------------------------------------

  [rank 1  distance 0.2189  id=25121  title=Software developer]
  SSOC Code: 25121
  Title: Software developer
  
  Definition:
  Software developer researches, designs and develops computer and network software or specialised utility programmes. He/she analyses user needs and develops intuitive and responsive software solutions, applying principles and techniques of computer science, engineering, and mathematical analysis. He/she also updates software, enhances existing software capabilities, and develops and directs software testing and validation procedures and evaluates and addresses security vulnerabilities through security tools. He/she may work with computer hardware engineers to integrate hardware and software systems, and develop specifications and p

## Failure post-mortem (answerable misses only)

For any answerable probe where the gold did NOT appear in top-5, show the gold's actual chunk text from the source so we can compare what the system *should* have surfaced vs. what it did.

In [7]:
misses = df[(df["category"] == "answerable") & (~df["hit@5"])]
if misses.empty:
    print("No answerable misses — every gold code was retrieved in top-5.")
else:
    chunks_path = ROOT / "data" / "processed" / "ssoc_chunks.jsonl"
    by_id = {}
    for line in chunks_path.read_text(encoding="utf-8").splitlines():
        c = json.loads(line)
        by_id[c["id"]] = c
    for _, row in misses.iterrows():
        gold_chunk = by_id.get(row["gold"])
        print("=" * 100)
        print(f"MISS: query={row['query']!r}")
        print(f"  gold={row['gold']}  (title='{gold_chunk['title']}')")
        print("  gold's actual chunk text:")
        print("  " + gold_chunk["text"].replace("\n", "\n  "))
        print("\n  what was actually returned (top 5):")
        for rank, (id_, dist, title) in enumerate(
            zip(row["top5_ids"], row["top5_distances"], row["top5_titles"]), start=1,
        ):
            print(f"    [{rank}] dist={dist:.4f}  id={id_}  title={title}")
        print()

No answerable misses — every gold code was retrieved in top-5.


## Determinism check

Re-runs probe 1 twice and asserts identical results.

**Why retrieval is deterministic:**
1. `gemini-embedding-001` has no temperature/sampling — the same text yields the same vector (within floating-point tolerance; exact in practice).
2. Chroma's HNSW index, once built, returns the same top-k for the same query vector.
3. The chunks in `chroma_db/` are static (we don't modify the collection between queries).

So re-running this entire notebook without changes yields identical results.

In [8]:
q = PROBES[0][0]
a = col.query(query_embeddings=[embed_query(q)], n_results=5, include=["distances"])
b = col.query(query_embeddings=[embed_query(q)], n_results=5, include=["distances"])
assert a["ids"] == b["ids"], "non-deterministic ids"
assert a["distances"] == b["distances"], "non-deterministic distances"
print(f"determinism check passed")
print(f"  ids:       {a['ids'][0]}")
print(f"  distances: {[round(d, 6) for d in a['distances'][0]]}")

determinism check passed
  ids:       ['25121', '2519', '2512', '25141', '25190']
  distances: [0.218915, 0.220599, 0.231393, 0.237489, 0.238729]


## Scope

This notebook was built **solely to verify the retrieval quality of the embeddings** — i.e., whether the right SSOC chunk(s) actually surface for a given query. It does NOT evaluate generation quality (answer correctness, hallucination, abstention behaviour); that is the job of Phase 3 and the formal eval harness. The goal is to separate the two failure modes so we know whether observed end-to-end errors come from retrieval or from generation.

## Results

### Headline metrics (answerable subset — 15 probes)

| Metric | Value |
|---|---|
| recall@1 | 11/15 = **73%** |
| recall@4 | 15/15 = **100%** ✅ |
| recall@5 | 15/15 = **100%** |
| MRR | **0.844** |

### Top-1 distance distribution (by category)

| Category | n | min | median | mean | max |
|---|---|---|---|---|---|
| answerable | 15 | 0.174 | 0.218 | 0.225 | 0.283 |
| unanswerable | 6 | 0.270 | 0.356 | 0.349 | 0.426 |
| edge | 4 | 0.332 | 0.337 | 0.342 | 0.361 |

→ Medians clearly separable (0.218 vs 0.356). Answerable-max and unanswerable-min touch in a narrow overlap zone (~0.27–0.28). Useful as a complementary signal for threshold-based abstention; not enough on its own.

### Determinism

Re-querying probe 1 twice produced **identical ids and identical 6-decimal distances**. Embedding model has no sampling, HNSW once-built is deterministic, chunks are static — re-running the notebook yields the same results.

## Key observations

1. **`hit@k = False` for unanswerable & edge rows is *vacuously* False — not a quality verdict.** The columns are computed `bool(gold and gold in ids[:k])`; when `gold = None` (every unanswerable + edge probe), the short-circuit forces `False`. There is no "gold" to hit against, so the binary classification simply doesn't apply. Example: "machine operator" returned `[81830, 8160, 8131, 81311, 81312]` — all major-group-8 codes (qualitatively correct), but `hit@k = False` only because no single gold was labelled. Quality on those rows is judged by reading the chunk dump in cell 12. Headline metrics (cell 8) filter to answerable rows only, so they're unaffected.

2. **recall@1 misses are parent ↔ child or sibling confusions, not bugs.** All 4 misses (Member of Parliament → parent unit group `1111`; chef in restaurant → parent `3434`; "person writes software" → sibling `25141`; "unit group for software developers" → sibling `2519`) are codes whose definitions overlap with the gold by design. Inherent to chunking both 4-digit unit groups and 5-digit detailed occupations. With `k = 4` the gold is always in the context window, so the generator will still see it.

3. **Typo robustness** — "electrcian" (misspelt) still surfaced `74110` at rank 1 (distance 0.334, higher than 0.201 for the clean spelling, but correct).

4. **Highest-risk unanswerable: the obsolete-version query.** "What was the SSOC code for IT support in SSOC 2010?" → top-1 = `35123` (current IT support code), distance **0.274**, *inside* the answerable/unanswerable overlap zone. The system semantically surfaces a relevant *current-day* code, so distance is low. This is exactly the case where prompt-based abstention (using the report chunks about SSOC-2024-vs-2020) will need to do work that retrieval alone cannot.

## Verdict

**recall@4 = 100% ≥ 80% → green light to Phase 3 (Retrieve & Generate).**

No retrieval tuning required. The embedding + cosine retrieval is good enough that for every answerable query the gold chunk is in the top-4 context window. Remaining failure modes the eval will surface in Phase 3 will be **generation-side** (LLM picking parent over child, ignoring abstention rules, etc.), which is the right place to be when starting prompt work.